Testing Purposes

In [1]:
!pip install numpy gdal earthpy matplotlib torch torchvision torchaudio scikit-learn opencv-python tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 99.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
 

In [2]:
# Imports

import os
import random

import numpy as np


from osgeo import gdal, gdal_array
import earthpy.plot as ep

import matplotlib.pyplot as plt

import torch
import torch.optim as optim
import torch.nn as nn
import torchvision.models as models

from sklearn.model_selection import train_test_split

import cv2 as cv

import tqdm


In [3]:
# from google.colab import drive
# import shutil
# import tarfile
# import gdown

# #
# # # Mount Google Drive
# # drive.mount('/content/drive')
# #
# # # Download the tar file from Google Drive using gdown
# # file_id = '1-cU2qx7XY_lwCC7PKOnnNRkeyRto80gC'
# # gdown.download(f'https://drive.usercontent.google.com/download?id={file_id}', 'dataset.tar', quiet=False)
# # https://drive.usercontent.google.com/download?id=1-cU2qx7XY_lwCC7PKOnnNRkeyRto80gC

# !curl 'https://drive.usercontent.google.com/download?id=1-cU2qx7XY_lwCC7PKOnnNRkeyRto80gC&confirm=t&uuid=40fa92f9-cb61-409f-9cc4-fa1177808d22&at=APcmpozDd3cdEoYOU-SWmN3eG3Eg%3A1745677659969' -H 'User-Agent: Mozilla/5.0 (X11; Linux x86_64; rv:136.0) Gecko/20100101 Firefox/136.0' -H 'Accept: text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8' -H 'Accept-Language: en-US,en;q=0.5' -H 'Accept-Encoding: gzip, deflate, br, zstd' -H 'DNT: 1' -H 'Alt-Used: drive.usercontent.google.com' -H 'Connection: keep-alive' -H 'Referer: https://drive.usercontent.google.com/' -H 'Cookie: AEC=AVcja2dufTQfITbDaB1d6L3pkeaqzR367oiRslPvSQxcOhvmXk2XzhTJm-I; NID=523=LZRksMjdfBJzFSR1mlmVIkMj-xIh2J6W8V7xuLmeKpl3Tw8f560KPxk5d7xkgL3UIMqfjxhislO4l6IY58e0piSjk9xIPhaacU8fi7Q6SJ87NtiKdMRn0L54MHgvx1wpf0p-VfvtXPf_BIsvk0E3Q0A5uBux94ZtI_MQKfLJlsQcdcPZzSgKd6eGHwt09YKsUFC5ByZtFY6cgAXAz756FScnb4JtwxV6RDVEF8LEWtwG3Qmw6XCp8lzl4OrbYsyG_9_vvpy2LEvPQoQ7Cw1upn2AJ2cgnuu6FIotkzPSxZGssqVZoIZM6lNkuchctd5R615-0z0Pd2Tb_i1DOdKKxuxZy-MrB0OvNxBYE5SvzxlXBtEeD5BArzFRX4xN6HXj2GuD693s3BubKLcsf6Y2tcX02Z3NN9rUMEquvmbethVCdrKKyYClzLXsmvI9KNteXX9UGUp2TOOiCpJu6xgr3eiCCylAUXdLZ9D1M6Kt8q02lFmM_rRjvUtj7Bt_zZ4qTrFC-4i6AoOqmxIepqnsRF_OPJTW_QkJgmI1ZmSraayKbCEZ9-L1rhyAtSz1F9Jtw1xn2QqZct_g4_A7ozQVi4BtiKwXbTySUagTG31_v9yItC_T0Er_NiG_F3bfnpq6Y70Xzj3Ilh1yR7YXceJXF54nT_qE1N73l-u9GDydF6oSsU1vnFxKiDfQ3-MacRPC62rMLgNfFE3T1lBM5iOdHOKV64pdyWknio9TMY8xqBJEqTiwx_V1WIPM9LEGNnOLr3z5dFBxhQDoaphubLwj6eETRgn9Z0135G5_5f_AJuoyb4X6QGQGGF8t8HcEDa7x-7medPnn5Ajcbyfs93z3UKPMwBSL58mqH6qz2Ul_tG7IG8aR5i7lqqJPeUpW1JwNTnd9LuuuxwSBYZ6UNPxxmLFiiJPh3ZtjgjnU0P2dNbwkA-a82WaIjdjB7-0KeB2uCf9XkpdxAgm3PyFvv-mRmu5Yk8eAtioRPVki_QMW7SBpZrEoq1_xIG1tLZQcLbPjJ5SoVNimo3_BtAlp5_LHJTbywf-Eb7VTHns2glF6u37LvLTL3gBQrttN1_mk3Ho7yJuuzmlNIAd2Y1Ocnjc2AYDLGo7nTEFrirAw6EIZkobnuLyg4ZRLlz_-Eybch-v22XOX_a2ZUl1n31im2hsoe7QLTGo10cuSyuI6I_ILqMcEyBpL-i8XLoubuaTbZd0v5hs2v27J3UpqZWPhpMH0W88AqZveebBbIOu8l3t9QBkwOj7xKObZSUdnKkQTqKXS2QlULY-EecXJXCmaM1yLTUBnXJVDjMOqrHYEtmZIF4lcID9n3TooAyZ18gq5OZCKf5ASYu-ulMefKQtg_YEmPZKGkMO3QePKKxD0T-jwlR6uofcivk_iraaPGnjJ8GW5ItYzmijjhD_g0A_cd7Be51UYCucs6aJr8vH-LktepL-cXagR-cZwKfxg4U6umrks9fA1FOebYNah-pfRFfTmvphcmkl46zcd8rmbWruYghKlcO2URZUMg9WMNiRBXfI7vV3D4gMgDUyPbvnkZV4AzW0hTGRDeSNhfbn1rjEYfDOUuR6VvRyosaFTyNh5BtAnNCSzV_YHHaJnWrLnBVcnGNMKfgdO4QntLVaWhy-2QlMIpGcPUtgTylq5mNn6INdHwrCNeqMLhohW1IYm0SLWdoGUfLZn79VL; SID=g.a000wQjK_2cs_vRMylSvNK8VFBW_nJZToVPUaIx6_qpTXbLj-ZLx6JoHXacruH8MfTR1BmEf6QACgYKAVcSARcSFQHGX2MiUN4TYaPYhQwxXYHheRnWCRoVAUF8yKrTrjLbDMdgFlMloYFxWmzG0076; __Secure-1PSID=g.a000wQjK_2cs_vRMylSvNK8VFBW_nJZToVPUaIx6_qpTXbLj-ZLxpRjCVRLfDR0HpqWlcV6pOgACgYKAS4SARcSFQHGX2MiBo1J3HuR4ExSFLm5a4RjlBoVAUF8yKoE6ffZumlz3EazTCxgjs7E0076; __Secure-3PSID=g.a000wQjK_2cs_vRMylSvNK8VFBW_nJZToVPUaIx6_qpTXbLj-ZLxTfjI1pIcV6SPHdNfWS66DgACgYKAc4SARcSFQHGX2Mi8pN1RHp3vS4Mhz3GXnBTUBoVAUF8yKrJXp3tcSlMPRU16mBGYCZT0076; HSID=AOhYaei00Xrot3bBq; SSID=ApB842gNUgc7hOEmi; APISID=n4N8jvcy0WsAek70/Aeh9r3_-6F86LMi2g; SAPISID=d438WokxuIBq0PCv/A6lt6jPJB7iRAH7vi; __Secure-1PAPISID=d438WokxuIBq0PCv/A6lt6jPJB7iRAH7vi; __Secure-3PAPISID=d438WokxuIBq0PCv/A6lt6jPJB7iRAH7vi; SIDCC=AKEyXzWbjC9Re6FmIggY98_8GHagDPnDIymut7fgogeweb1Vg7XCIweQp4GOKrZ5s-BwdS6G9N8; __Secure-1PSIDCC=AKEyXzXXQUtqCtmUZCGL8vs0Y0agtP-KBnXR1TlVkbbyC_iQZIgToAlHL9K20M6FJzlZpnt_7VQ; __Secure-3PSIDCC=AKEyXzW7-r-ROrqD-YcGLIQ2o9ohb5bRW9huolf-Q57cwIdFTLHRSn3BmEYt952Jt6vYWWGb93g; __Secure-1PSIDTS=sidts-CjIBjplskO740q3pF-rLlU51_X-jdQ039G-0uwFniQTl4pMCYyVaMeR8pS31NeiOKAMgqhAA; __Secure-3PSIDTS=sidts-CjIBjplskO740q3pF-rLlU51_X-jdQ039G-0uwFniQTl4pMCYyVaMeR8pS31NeiOKAMgqhAA' -H 'Upgrade-Insecure-Requests: 1' -H 'Sec-Fetch-Dest: document' -H 'Sec-Fetch-Mode: navigate' -H 'Sec-Fetch-Site: cross-site' -H 'Sec-Fetch-User: ?1' -H 'Priority: u=0, i' -H 'Pragma: no-cache' -H 'Cache-Control: no-cache' -H 'TE: trailers' --output dataset.tar

# # Extract the tar file
# with tarfile.open('dataset.tar', 'r') as tar:
#     tar.extractall(path='/content')  # Specify folder name to extract to

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 23.2G  100 23.2G    0     0  72.3M      0  0:05:29  0:05:29 --:--:-- 95.5M


In [4]:
content_dir = os.path.join("content", "train")
image_base_dir = os.path.join(content_dir, "data")
mask_base_dir = os.path.join(content_dir, "masks")

# Check if image directory exists
if not os.path.exists(image_base_dir):
    raise FileNotFoundError(f"Data Doesn't Exist: {image_base_dir}")

# List and sort files to maintain consistent ordering
image_paths = sorted([
    os.path.join(image_base_dir, fname)
    for fname in os.listdir(image_base_dir) if fname.endswith(".tif")
])

mask_paths = sorted([
    os.path.join(mask_base_dir, fname)
    for fname in os.listdir(mask_base_dir) if fname.endswith(".tif")
])

# Ensure matching counts
assert len(image_paths) == len(mask_paths), "Mismatch between image and mask counts"

# Optional: shuffle deterministically
combined = list(zip(image_paths, mask_paths))
random.seed(42)
random.shuffle(combined)
image_paths, mask_paths = zip(*combined)

# Split 60% train, 20% val, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(image_paths, mask_paths, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

# Now you have:
print(f"Train: {len(X_train)} images")
print(f"Validation: {len(X_val)} images")
print(f"Test: {len(X_test)} images")

Train: 6343 images
Validation: 2115 images
Test: 2115 images


In [5]:
def convert_to_nparr(dataset):
    dtype = gdal_array.GDALTypeCodeToNumericTypeCode(dataset.GetRasterBand(1).DataType)
    arr = np.zeros((dataset.RasterYSize, dataset.RasterXSize, dataset.RasterCount), dtype=dtype)
    bands = []
    for i in range(dataset.RasterCount):
        arr[:, :, i] = dataset.GetRasterBand(i + 1).ReadAsArray()
        bands.append(arr[:, :, i])
    bands = np.stack(bands)
    return bands

In [ ]:
# ===================================== VISUALIZATION =====================================

# # Set a seed for reproducibility
# seed = 0
# random.seed(seed)

# # Select 25 random images
# random_images = random.sample(satellite_images, 1)



# for i, image_path in enumerate(random_images):
#     base_image = os.path.basename(image_path)
#     mask_path = os.path.join(mask_base_dir, base_image)

#     satellite_image = gdal.Open(image_path)
#     mask_image = gdal.Open(mask_path)

#     satellite_image = convert_to_nparr(satellite_image)
#     mask_image = convert_to_nparr(mask_image)

#     print(f"Image shape: {satellite_image.shape}")  # (bands, height, width)

#     # Also works with earthpy
#     ep.plot_rgb(satellite_image, rgb=(3, 2, 1), title=f"{image_path}")


In [6]:
# Data Loaders

class SatelliteDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        image = gdal.Open(image_path)
        mask = gdal.Open(mask_path)

        if image is None:
            raise ValueError(f"Failed to load image at {image_path}")
        if mask is None:
            raise ValueError(f"Failed to load mask at {mask_path}")

        image = convert_to_nparr(image)
        mask = convert_to_nparr(mask)

        image = torch.from_numpy(image).float()
        mask = torch.from_numpy(mask).float()

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        return image, mask



In [12]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=False, batchnorm=True, dropout_prob=0.5):
        super(ConvBlock, self).__init__()
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels) if batchnorm else nn.Identity(),
            nn.ReLU(inplace=True),
        ]
        layers += [
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels) if batchnorm else nn.Identity(),
            nn.ReLU(inplace=True),
        ]
        if dropout:
            layers.append(nn.Dropout(dropout_prob))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)



class UNet(nn.Module):
    in_channels = 4

    def prepare_image(self, x):
        return x

    def __init__(self, out_channels, dropout_prob=0.5):
        super(UNet, self).__init__()

        resnet = models.resnet18(pretrained=True)

        # If input has 4 channels, adapt first conv
        if self.in_channels != 3:
            original_conv = resnet.conv1
            self.resnet_conv1 = nn.Conv2d(
                self.in_channels,
                original_conv.out_channels,
                kernel_size=original_conv.kernel_size,
                stride=original_conv.stride,
                padding=original_conv.padding,
                bias=original_conv.bias is not None
            )
            with torch.no_grad():
                self.resnet_conv1.weight[:, :3] = original_conv.weight
                self.resnet_conv1.weight[:, 3:] = original_conv.weight[:, :1]
        else:
            self.resnet_conv1 = resnet.conv1

        self.resnet_bn1 = resnet.bn1
        self.resnet_relu = resnet.relu
        self.resnet_pool = resnet.maxpool  # Keep ResNet's own pool

        # Encoder
        self.enc2 = ConvBlock(64, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = ConvBlock(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.enc4 = ConvBlock(128, 256)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(256, 512)

        # Decoder
        self.upconv4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec4 = ConvBlock(512, 256, batchnorm=False)

        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(256, 128, batchnorm=False)

        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(128, 64, batchnorm=False)

        self.upconv1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(128, 32, dropout=True, batchnorm=False, dropout_prob=dropout_prob)

        # **NEW final upsampling layer**
        self.final_upconv = nn.ConvTranspose2d(32, 32, kernel_size=2, stride=2)

        self.out_conv = nn.Conv2d(32, out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.prepare_image(x)

        enc1 = self.resnet_conv1(x)
        enc1 = self.resnet_bn1(enc1)
        enc1 = self.resnet_relu(enc1)
        enc1_pool = self.resnet_pool(enc1)  # Use ResNet's own maxpool, not extra pooling

        enc2 = self.enc2(enc1_pool)
        enc2_pool = self.pool2(enc2)

        enc3 = self.enc3(enc2_pool)
        enc3_pool = self.pool3(enc3)

        enc4 = self.enc4(enc3_pool)
        enc4_pool = self.pool4(enc4)

        bottleneck = self.bottleneck(enc4_pool)

        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.dec4(dec4)

        dec3 = self.upconv3(dec4)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.dec3(dec3)

        dec2 = self.upconv2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.dec2(dec2)

        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.dec1(dec1)

        dec1 = self.final_upconv(dec1)

        dec0 = self.out_conv(dec1)
        out = self.sigmoid(dec0)
        return out


In [8]:
def dice_coefficient(pred_mask: torch.Tensor, true_mask: torch.Tensor, eps=1e-6) -> float:
    intersection = (pred_mask * true_mask).sum()
    total_pixels = pred_mask.sum() + true_mask.sum()
    dice = (2.0 * intersection + eps) / (total_pixels + eps)
    return dice.item()

In [15]:
# Hyperparameters

BATCH_SIZE = 16
DROPOUT_PROB = 0.4
LEARNING_RATE = 0.001
NUM_EPOCHS = 10
CRITERION = nn.BCEWithLogitsLoss()


In [13]:
# Data Loaders

from torchvision import transforms

# Define the transform for training images
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=45),
])

train_dataset = SatelliteDataset(X_train, y_train, transform=train_transform)
val_dataset = SatelliteDataset(X_val, y_val)
test_dataset = SatelliteDataset(X_test, y_test)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(out_channels=1, dropout_prob=DROPOUT_PROB).to(device)  # 1 output channel for binary mask

criterion = CRITERION
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

In [ ]:
print(DROPOUT_PROB)
for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0

    for img, mask in tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        # print(f"Image shape: {img.shape}")  # (batch_size, bands, height, width)
        # print(f"Mask shape: {mask.shape}")

        img = img.to(device)
        mask = mask.to(device)

        # Forward pass
        outputs = model(img)
        loss = criterion(outputs, mask)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation (optional, but good practice)
    model.eval()
    total_dice = 0.0
    with torch.no_grad():
        for val_image, val_mask in val_loader:
            val_image, val_mask = val_image.to(device), val_mask.to(device)
            pred_mask = model(val_image)

            pred_mask = (pred_mask > 0.5).float()
            val_mask = (val_mask > 0.5).float()

            total_dice += dice_coefficient(pred_mask, val_mask)



    val_loss = total_dice / len(val_loader)

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] - Train Loss: {train_loss:.4f} - Val Acc: {val_loss:.4f}")

0.4


Epoch 1/10: 100%|██████████| 397/397 [07:52<00:00,  1.19s/it]


Epoch [1/10] - Train Loss: 0.6931 - Val Acc: 0.6818


Epoch 2/10: 100%|██████████| 397/397 [07:49<00:00,  1.18s/it]


Epoch [2/10] - Train Loss: 0.6916 - Val Acc: 0.7716


...

Epoch 5/10: 100%|██████████| 397/397 [08:08<00:00,  1.23s/it]


Epoch [5/10] - Train Loss: 0.6884 - Val Acc: 0.7716


Epoch 6/10: 100%|██████████| 397/397 [08:07<00:00,  1.23s/it]


Epoch [6/10] - Train Loss: 0.6877 - Val Acc: 0.7716


Epoch 7/10:  29%|██▉       | 116/397 [02:22<05:36,  1.20s/it]

In [ ]:
# import torch
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()
# !nvidia-smi | grep python
# # !kill -9 $(nvidia-smi | grep python | awk '{print $5}')
# !pkill -9 python


In [ ]:
# Save both the model's state_dict and the full model
torch.save(model.state_dict(), 'model_state_dict.pth')  # Recommended way
torch.save(model, 'model_full.pth')                     # Full model (less portable)

In [ ]:
# Load the model
model = UNet(out_channels=1)
model.load_state_dict(torch.load('model_state_dict.pth'))
model.eval()  # Set the model to evaluation mode
# Test the model
model.to("cuda")
total_dice = 0.0
with torch.no_grad():
    for test_images, test_masks in test_loader:
        test_images, test_masks = test_images.to(device), test_masks.to(device)

        # print(test_images.shape)

        pred_masks = model(test_images)

        pred_masks = (pred_masks > 0.5).float()
        test_masks = (test_masks > 0.5).float()

        total_dice += dice_coefficient(pred_masks, test_masks)

    test_dice = total_dice / len(test_loader)
    print(f"Test Dice Coefficient: {test_dice:.4f}")

In [ ]:
from evaluate_pref import profile

# Load the model
model = UNet(out_channels=1)
model.load_state_dict(torch.load('model_state_dict.pth'))

num_ops, num_params = profile(model, (16, 4, 512, 512))

print(f"Number of operations: {num_ops}")
print(f"Number of parameters: {num_params}")